<a href="https://colab.research.google.com/github/plnu-biomechanics/kin4042/blob/main/kin4042_lab2_key.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://www.pointloma.edu/sites/default/files/styles/basic_page/public/images/PLNU_Biomechanics_Lab_green_yellowSD_HiRes.png" width="400">

# KIN 4042 Sport Informatics
## Lab 2: Between-session reliability of CMJ performance and asymmetry

**Instructor:** Arnel Aguinaldo, PhD

In this lab, you will reproduce the principal summary and reliability analyses from:

Pérez-Castilla, A., García-Ramos, A., Janicijevic, D., Delgado-García, G., De la Cruz, J. C., Rojas, F. J., & Cepero, M. (2021). Between-session reliability of performance and asymmetry variables obtained during unilateral and bilateral countermovement jumps in basketball players. *PLOS ONE, 16*(7), e0255458. https://doi.org/10.1371/journal.pone.0255458

Twenty-three basketball players completed two identical CMJ testing sessions separated by seven days. The notebook uses the public, athlete-level supporting dataset.

## Learning objectives

By the end of this lab, you should be able to:

1. import and audit repeated-measures sport-performance data;
2. reshape a dataset between wide and long formats;
3. calculate signed inter-limb asymmetry;
4. summarize performance and asymmetry by session;
5. choose between a paired *t* test and Wilcoxon signed-rank test;
6. calculate Cohen's *d*, SEM-based coefficient of variation (CV), and ICC(3,1);
7. interpret absolute and relative reliability together.

> **Replication note:** The public CSV contains rounded values. Small differences from the published table are expected. It also does not identify each player's preferred leg. Therefore, unilateral asymmetry can be reproduced directly, whereas the paper's preference-leg coding for bilateral asymmetry cannot be reconstructed exactly.

## 1. Import the Python libraries

In [1]:
# Core libraries available in Google Colab; no additional installation is required.
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.metrics import cohen_kappa_score
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid", context="notebook")

## 2. Download and load the data

The code downloads the CSV into the temporary Colab runtime. If the GitHub URL is unavailable, upload the CSV into the same folder and rerun the loading cell.

In [2]:
data_dir = Path("kin4042/lab2")
data_dir.mkdir(parents=True, exist_ok=True)

data_url = (
    "https://raw.githubusercontent.com/plnu-biomechanics/"
    "kin4042/refs/heads/main/labs/pone_0255458_s001.csv"
)
data_path = data_dir / "pone_0255458_s001.csv"

urllib.request.urlretrieve(data_url, data_path)
print(f"Downloaded: {data_path}")

Downloaded: kin4042/lab2/pone_0255458_s001.csv


In [3]:
# utf-8-sig safely removes the byte-order mark found in the original CSV.
cmj = pd.read_csv(data_path, encoding="utf-8-sig")

# Replace hyphens in column names such as CI-UNICMJ_RIGHT.
cmj.columns = cmj.columns.str.replace("-", "_", regex=False)
cmj = cmj.sort_values(["Code", "Session"]).reset_index(drop=True)

print(f"Shape: {cmj.shape}")
display(cmj.head())

Shape: (46, 32)


,Code,Session,MF_UNICMJ_RIGHT,PF_UNICMJ_RIGHT,MV_UNICMJ_RIGHT,PV_UNICMJ_RIGHT,MP_UNICMJ_RIGHT,PP_UNICMJ_RIGHT,CI_UNICMJ_RIGHT,JH_UNICMJ_RIGHT,MF_UNICMJ_LEFT,PF_UNICMJ_LEFT,MV_UNICMJ_LEFT,PV_UNICMJ_LEFT,MP_UNICMJ_LEFT,PP_UNICMJ_LEFT,CI_UNICMJ_LEFT,JH_UNICMJ_LEFT,MF_BICMJ_RIGHT,PF_BICMJ_RIGHT,MV_BICMJ_RIGHT,PV_BICMJ_RIGHT,MP_BICMJ_RIGHT,PP_BICMJ_RIGHT,CI_BICMJ_RIGHT,MF_BICMJ_LEFT,PF_BICMJ_LEFT,MV_BICMJ_LEFT,PV_BICMJ_LEFT,MP_BICMJ_LEFT,PP_BICMJ_LEFT,CI_BICMJ_LEFT
0,1,1,1126.2,1379.4,1.22,2.10,1254.2,2396.2,134.1,0.19,1122.7,1388.6,1.21,2.09,1233.7,2421.5,130.2,0.18,666.4,786.9,1.55,2.75,952.2,1891.2,94.2,616.7,756.4,1.06,1.87,600.4,1245.0,65.8
1,1,2,1179.8,1617.9,1.24,2.00,1278.3,2497.3,121.1,0.14,1126.6,1479.2,1.16,1.91,1161.9,2173.2,115.8,0.15,636.2,808.3,1.60,2.90,939.9,2019.1,94.0,564.7,735.8,0.71,1.27,367.6,796.6,41.5
2,2,1,1127.5,1308.5,1.20,2.06,1242.9,2329.0,138.4,0.15,1104.2,1354.7,1.10,2.01,1141.5,2342.8,132.8,0.18,850.8,1025.2,1.76,2.92,1329.6,2194.8,109.4,676.6,831.9,1.65,2.89,1027.4,1862.0,102.0
3,2,2,1239.9,1479.2,1.20,2.03,1348.4,2470.2,139.1,0.22,1338.7,1602.0,1.16,2.05,1373.6,2546.1,157.6,0.18,891.7,1081.1,1.72,2.87,1366.8,2284.5,109.7,802.7,992.5,1.60,2.68,1143.3,1932.1,94.6
4,3,1,1034.0,1265.5,1.29,2.15,1224.2,2036.3,130.6,0.25,1029.3,1176.2,1.30,2.33,1254.6,2357.1,140.9,0.25,653.4,821.2,1.88,3.18,1117.0,1843.5,102.1,643.1,773.7,1.40,2.67,813.2,1621.6,92.1


### Data audit

In [4]:
sessions_per_player = cmj.groupby("Code")["Session"].nunique()

audit = pd.Series({
    "rows": len(cmj),
    "columns": cmj.shape[1],
    "unique_players": cmj["Code"].nunique(),
    "unique_sessions": cmj["Session"].nunique(),
    "missing_cells": int(cmj.isna().sum().sum()),
    "players_with_two_sessions": int((sessions_per_player == 2).sum()),
}, name="Value").to_frame()

display(audit)

assert len(cmj) == 46, "Expected 46 rows."
assert cmj["Code"].nunique() == 23, "Expected 23 players."
assert set(cmj["Session"].unique()) == {1, 2}, "Expected sessions 1 and 2."
assert sessions_per_player.eq(2).all(), "Every player must have two sessions."
assert cmj.isna().sum().sum() == 0, "Unexpected missing data found."

print("Data audit passed.")

,Value
rows,46
columns,32
unique_players,23
unique_sessions,2
missing_cells,0
players_with_two_sessions,23


Data audit passed.


## 3. Reshape and summarize the performance variables

Variable prefixes:

| Prefix | Metric | Unit |
|---|---|---|
| MF | Mean force | N |
| PF | Peak force | N |
| MV | Mean velocity | m/s |
| PV | Peak velocity | m/s |
| MP | Mean power | W |
| PP | Peak power | W |
| CI | Concentric impulse | N·s |
| JH | Jump height | m |

`UNICMJ` denotes a unilateral CMJ; `BICMJ` denotes a bilateral CMJ. The final component identifies the right or left limb.

In [ ]:
id_columns = ["Code", "Session"]
performance_columns = [column for column in cmj.columns if column not in id_columns]

performance_long = cmj.melt(
    id_vars=id_columns,
    value_vars=performance_columns,
    var_name="Variable",
    value_name="Value",
)

components = performance_long["Variable"].str.extract(
    r"^(?P<Metric>[A-Z]+)_(?P<Jump>UNICMJ|BICMJ)_(?P<Limb>RIGHT|LEFT)$"
)
performance_long = pd.concat([performance_long, components], axis=1)
performance_long["Jump"] = performance_long["Jump"].map({
    "UNICMJ": "Unilateral CMJ",
    "BICMJ": "Bilateral CMJ",
})
performance_long["Limb"] = performance_long["Limb"].str.title()

display(performance_long.head(8))

In [ ]:
performance_summary = (
    performance_long
    .groupby(["Jump", "Metric", "Limb", "Session"], as_index=False)
    .agg(
        n=("Value", "count"),
        Mean=("Value", "mean"),
        SD=("Value", "std"),
        Minimum=("Value", "min"),
        Maximum=("Value", "max"),
    )
)

display(
    performance_summary.style
    .format({"Mean": "{:.2f}", "SD": "{:.2f}", "Minimum": "{:.2f}", "Maximum": "{:.2f}"})
    .set_caption("CMJ performance by session")
)

### Visualize paired session values

In [ ]:
# Change this string to inspect a different variable.
selected_variable = "JH_UNICMJ_RIGHT"

selected = cmj[["Code", "Session", selected_variable]].copy()

fig, ax = plt.subplots(figsize=(7, 5))
for _, player in selected.groupby("Code"):
    ax.plot(player["Session"], player[selected_variable], color="0.70", alpha=0.65, linewidth=1)
ax.scatter(selected["Session"], selected[selected_variable], c=selected["Session"],
           cmap="viridis", s=42, zorder=3)
ax.set(xticks=[1, 2], xlabel="Session", ylabel=selected_variable,
       title=f"Paired values: {selected_variable}")
plt.show()

## 4. Calculate inter-limb asymmetry

For unilateral CMJs, the article used the percentage-difference formula

$$
100\left(1-\frac{\min(R,L)}{\max(R,L)}\right).
$$

We add a sign so that positive scores indicate a larger right-leg value and negative scores indicate a larger left-leg value:

$$
\text{signed unilateral asymmetry}=100\frac{R-L}{\max(R,L)}.
$$

In [ ]:
def signed_unilateral_asymmetry(right, left):
    # Signed percentage difference: positive favors right, negative favors left.
    denominator = np.maximum(right, left)
    return np.where(denominator == 0, np.nan, 100 * (right - left) / denominator)


def bilateral_asymmetry_magnitude(right, left):
    # Unsigned bilateral asymmetry magnitude; does not require preferred leg.
    denominator = right + left
    return np.where(denominator == 0, np.nan, 100 * np.abs(right - left) / denominator)


metric_codes = ["MF", "PF", "MV", "PV", "MP", "PP", "CI"]
cmj_analysis = cmj.copy()

for metric in metric_codes:
    cmj_analysis[f"ASYM_{metric}_UNICMJ"] = signed_unilateral_asymmetry(
        cmj_analysis[f"{metric}_UNICMJ_RIGHT"],
        cmj_analysis[f"{metric}_UNICMJ_LEFT"],
    )
    cmj_analysis[f"ASYM_{metric}_BICMJ_MAG"] = bilateral_asymmetry_magnitude(
        cmj_analysis[f"{metric}_BICMJ_RIGHT"],
        cmj_analysis[f"{metric}_BICMJ_LEFT"],
    )

# Jump height was reported only for unilateral CMJs.
cmj_analysis["ASYM_JH_UNICMJ"] = signed_unilateral_asymmetry(
    cmj_analysis["JH_UNICMJ_RIGHT"],
    cmj_analysis["JH_UNICMJ_LEFT"],
)

unilateral_asymmetry_columns = [
    column for column in cmj_analysis.columns
    if column.startswith("ASYM_") and column.endswith("_UNICMJ")
]
bilateral_magnitude_columns = [
    column for column in cmj_analysis.columns
    if column.startswith("ASYM_") and column.endswith("_BICMJ_MAG")
]

display(cmj_analysis[["Code", "Session"] + unilateral_asymmetry_columns].head())

In [ ]:
asymmetry_long = cmj_analysis.melt(
    id_vars=["Code", "Session"],
    value_vars=unilateral_asymmetry_columns + bilateral_magnitude_columns,
    var_name="Variable",
    value_name="Asymmetry",
)
asymmetry_long["Analysis"] = np.where(
    asymmetry_long["Variable"].str.endswith("_BICMJ_MAG"),
    "Bilateral magnitude (extension)",
    "Unilateral signed asymmetry",
)
asymmetry_long["Metric"] = asymmetry_long["Variable"].str.extract(r"^ASYM_([A-Z]+)_")

asymmetry_summary = (
    asymmetry_long
    .groupby(["Analysis", "Metric", "Session"], as_index=False)
    .agg(
        n=("Asymmetry", "count"),
        Mean=("Asymmetry", "mean"),
        SD=("Asymmetry", "std"),
        Minimum=("Asymmetry", "min"),
        Maximum=("Asymmetry", "max"),
    )
)

display(
    asymmetry_summary.style
    .format({"Mean": "{:.2f}", "SD": "{:.2f}", "Minimum": "{:.2f}", "Maximum": "{:.2f}"})
    .set_caption("Inter-limb asymmetry by session")
)

## 5. Reliability-analysis functions

For each variable, we will calculate:

- a paired *t* test when paired differences are normally distributed, or a Wilcoxon signed-rank test otherwise;
- Cohen's *d* using $(M_2-M_1)/SD_{pooled}$, matching the article's definition;
- SEM and CV from the residual error of a two-way subject-by-session ANOVA;
- ICC(3,1): two-way mixed-effects, consistency, single measure;
- 95% confidence intervals for CV and ICC.

The article classified a performance metric as acceptable when **ICC > 0.70 and CV < 10%**. CV is not used for signed asymmetry because its mean can approach zero.

In [ ]:
def make_pairs(data, variable):
    # Return complete Session 1 and Session 2 arrays aligned by player.
    paired = (
        data[["Code", "Session", variable]]
        .pivot(index="Code", columns="Session", values=variable)
        .dropna(subset=[1, 2])
        .sort_index()
    )
    return paired[1].to_numpy(float), paired[2].to_numpy(float)


def icc_3_1(session_1, session_2, alpha=0.05):
    # ICC(3,1): two-way mixed, consistency, single-measure ICC.
    values = np.column_stack([session_1, session_2])
    n, k = values.shape
    grand_mean = values.mean()
    subject_means = values.mean(axis=1)
    session_means = values.mean(axis=0)

    ss_subject = k * np.sum((subject_means - grand_mean) ** 2)
    residuals = values - subject_means[:, None] - session_means[None, :] + grand_mean
    ss_error = np.sum(residuals ** 2)

    df_subject = n - 1
    df_error = (n - 1) * (k - 1)
    ms_subject = ss_subject / df_subject
    ms_error = ss_error / df_error

    icc = (ms_subject - ms_error) / (ms_subject + (k - 1) * ms_error)

    # F-based confidence interval for ICC(C,1). Spreadsheet/package conventions
    # can produce slightly different limits, but the point estimate is identical.
    f_ratio = ms_subject / ms_error
    f_lower = f_ratio / stats.f.ppf(1 - alpha / 2, df_subject, df_error)
    f_upper = f_ratio * stats.f.ppf(1 - alpha / 2, df_error, df_subject)
    icc_low = (f_lower - 1) / (f_lower + k - 1)
    icc_high = (f_upper - 1) / (f_upper + k - 1)

    return {
        "ICC_3_1": icc,
        "ICC_low": icc_low,
        "ICC_high": icc_high,
        "MS_error": ms_error,
        "df_error": df_error,
    }


def analyze_reliability(data, variable, report_cv=True):
    session_1, session_2 = make_pairs(data, variable)
    difference = session_2 - session_1
    n = len(difference)

    shapiro = stats.shapiro(difference)
    if shapiro.pvalue > 0.05:
        comparison = stats.ttest_rel(session_2, session_1)
        test_name = "Paired t test"
    else:
        comparison = stats.wilcoxon(session_2, session_1, method="approx", correction=False)
        test_name = "Wilcoxon signed-rank"

    pooled_sd = np.sqrt((np.var(session_1, ddof=1) + np.var(session_2, ddof=1)) / 2)
    cohens_d = (np.mean(session_2) - np.mean(session_1)) / pooled_sd

    icc = icc_3_1(session_1, session_2)
    sem = np.sqrt(icc["MS_error"])

    if report_cv:
        mean_score = np.mean(np.concatenate([session_1, session_2]))
        cv = 100 * sem / mean_score
        df_error = icc["df_error"]
        cv_low = cv * np.sqrt(df_error / stats.chi2.ppf(0.975, df_error))
        cv_high = cv * np.sqrt(df_error / stats.chi2.ppf(0.025, df_error))
    else:
        sem = cv = cv_low = cv_high = np.nan

    return {
        "Variable": variable,
        "n": n,
        "Session_1_Mean": np.mean(session_1),
        "Session_1_SD": np.std(session_1, ddof=1),
        "Session_2_Mean": np.mean(session_2),
        "Session_2_SD": np.std(session_2, ddof=1),
        "Test": test_name,
        "Normality_p": shapiro.pvalue,
        "Comparison_p": comparison.pvalue,
        "Cohens_d": cohens_d,
        "SEM": sem,
        "CV": cv,
        "CV_low": cv_low,
        "CV_high": cv_high,
        "ICC_3_1": icc["ICC_3_1"],
        "ICC_low": icc["ICC_low"],
        "ICC_high": icc["ICC_high"],
    }

## 6. Check normality of paired differences

A paired *t* test assumes that the **within-player differences** are normally distributed. It does not require each session by itself to be normal.

In [ ]:
analysis_variables = performance_columns + unilateral_asymmetry_columns

normality_results = []
for variable in analysis_variables:
    session_1, session_2 = make_pairs(cmj_analysis, variable)
    result = stats.shapiro(session_2 - session_1)
    normality_results.append({
        "Variable": variable,
        "n": len(session_1),
        "W": result.statistic,
        "p": result.pvalue,
        "Recommended_test": "Paired t test" if result.pvalue > 0.05 else "Wilcoxon signed-rank",
    })

normality_results = pd.DataFrame(normality_results)
display(
    normality_results.style
    .format({"W": "{:.3f}", "p": "{:.3f}"})
    .set_caption("Shapiro-Wilk tests of paired differences")
)

## 7. Performance reliability

In [ ]:
performance_reliability = pd.DataFrame([
    analyze_reliability(cmj_analysis, variable, report_cv=True)
    for variable in performance_columns
])

performance_reliability["Jump"] = np.select(
    [
        performance_reliability["Variable"].str.contains("_UNICMJ_"),
        performance_reliability["Variable"].str.contains("_BICMJ_"),
    ],
    ["Unilateral CMJ", "Bilateral CMJ"],
    default="Unknown",
)
performance_reliability["Limb"] = np.where(
    performance_reliability["Variable"].str.endswith("_RIGHT"), "Right", "Left"
)
performance_reliability["Metric"] = performance_reliability["Variable"].str.extract(r"^([A-Z]+)")
performance_reliability["Reliability"] = np.where(
    (performance_reliability["ICC_3_1"] > 0.70) & (performance_reliability["CV"] < 10),
    "Acceptable",
    "Review",
)

performance_table = performance_reliability[[
    "Jump", "Metric", "Limb", "n",
    "Session_1_Mean", "Session_1_SD", "Session_2_Mean", "Session_2_SD",
    "Test", "Comparison_p", "Cohens_d", "CV", "CV_low", "CV_high",
    "ICC_3_1", "ICC_low", "ICC_high", "Reliability",
]].copy()

display(
    performance_table.style
    .format({
        "Session_1_Mean": "{:.2f}", "Session_1_SD": "{:.2f}",
        "Session_2_Mean": "{:.2f}", "Session_2_SD": "{:.2f}",
        "Comparison_p": "{:.3f}", "Cohens_d": "{:.2f}",
        "CV": "{:.2f}", "CV_low": "{:.2f}", "CV_high": "{:.2f}",
        "ICC_3_1": "{:.2f}", "ICC_low": "{:.2f}", "ICC_high": "{:.2f}",
    })
    .map(lambda value: "background-color: #fff2cc" if value == "Review" else "",
         subset=["Reliability"])
    .set_caption("Between-session reliability of CMJ performance")
)

## 8. Unilateral asymmetry reliability

The paper reported ICC, but not CV, for asymmetry. CV would be unstable when the signed group mean is close to zero.

In [ ]:
asymmetry_reliability = pd.DataFrame([
    analyze_reliability(cmj_analysis, variable, report_cv=False)
    for variable in unilateral_asymmetry_columns
])
asymmetry_reliability["Metric"] = asymmetry_reliability["Variable"].str.extract(
    r"^ASYM_([A-Z]+)_"
)
asymmetry_reliability["ICC_interpretation"] = np.where(
    asymmetry_reliability["ICC_3_1"] > 0.70, "Acceptable", "Unacceptable"
)

asymmetry_table = asymmetry_reliability[[
    "Metric", "n", "Session_1_Mean", "Session_1_SD", "Session_2_Mean",
    "Session_2_SD", "Test", "Comparison_p", "Cohens_d",
    "ICC_3_1", "ICC_low", "ICC_high", "ICC_interpretation",
]]

display(
    asymmetry_table.style
    .format({
        "Session_1_Mean": "{:.2f}", "Session_1_SD": "{:.2f}",
        "Session_2_Mean": "{:.2f}", "Session_2_SD": "{:.2f}",
        "Comparison_p": "{:.3f}", "Cohens_d": "{:.2f}",
        "ICC_3_1": "{:.2f}", "ICC_low": "{:.2f}", "ICC_high": "{:.2f}",
    })
    .map(lambda value: "background-color: #fff2cc" if value == "Unacceptable" else "",
         subset=["ICC_interpretation"])
    .set_caption("Between-session reliability of unilateral CMJ asymmetry")
)

### Direction-of-asymmetry agreement

ICC evaluates the consistency of asymmetry **scores**. Cohen's kappa evaluates whether the same side was favored in both sessions. The article interpreted kappa as poor (≤0), slight (.01–.20), fair (.21–.40), moderate (.41–.60), substantial (.61–.80), or almost perfect (.81–.99).

In [ ]:
def interpret_kappa(value):
    if value <= 0.00:
        return "Poor"
    if value <= 0.20:
        return "Slight"
    if value <= 0.40:
        return "Fair"
    if value <= 0.60:
        return "Moderate"
    if value <= 0.80:
        return "Substantial"
    return "Almost perfect"


kappa_results = []
for variable in unilateral_asymmetry_columns:
    session_1, session_2 = make_pairs(cmj_analysis, variable)
    direction_1 = np.where(session_1 >= 0, "Right", "Left")
    direction_2 = np.where(session_2 >= 0, "Right", "Left")
    kappa = cohen_kappa_score(direction_1, direction_2)
    kappa_results.append({
        "Metric": variable.split("_")[1],
        "Kappa": kappa,
        "Agreement": interpret_kappa(kappa),
        "Same_direction_n": int(np.sum(direction_1 == direction_2)),
        "Total_n": len(direction_1),
    })

kappa_results = pd.DataFrame(kappa_results)
display(
    kappa_results.style
    .format({"Kappa": "{:.2f}"})
    .set_caption("Agreement in the direction of unilateral CMJ asymmetry")
)

## 9. Optional extension: bilateral asymmetry magnitude

The dataset does not contain preferred-leg identity, so this is an **unsigned magnitude analysis** rather than an exact reproduction of the paper's bilateral preference-leg analysis. Do not compare these ICCs directly with the paper's bilateral asymmetry ICCs.

In [ ]:
bilateral_magnitude_reliability = pd.DataFrame([
    analyze_reliability(cmj_analysis, variable, report_cv=False)
    for variable in bilateral_magnitude_columns
])
bilateral_magnitude_reliability["Metric"] = bilateral_magnitude_reliability["Variable"].str.extract(
    r"^ASYM_([A-Z]+)_"
)

display(
    bilateral_magnitude_reliability[[
        "Metric", "n", "Session_1_Mean", "Session_1_SD", "Session_2_Mean",
        "Session_2_SD", "Test", "Comparison_p", "Cohens_d",
        "ICC_3_1", "ICC_low", "ICC_high",
    ]].style
    .format({
        "Session_1_Mean": "{:.2f}", "Session_1_SD": "{:.2f}",
        "Session_2_Mean": "{:.2f}", "Session_2_SD": "{:.2f}",
        "Comparison_p": "{:.3f}", "Cohens_d": "{:.2f}",
        "ICC_3_1": "{:.2f}", "ICC_low": "{:.2f}", "ICC_high": "{:.2f}",
    })
    .set_caption("Extension: bilateral CMJ asymmetry magnitude")
)

## 10. Verification checks

In [ ]:
# These checks verify the analysis against values independently calculated from
# the public CSV. They prevent silent errors if the code is modified.
check_right = performance_reliability.loc[
    performance_reliability["Variable"] == "MF_UNICMJ_RIGHT"
].iloc[0]
check_left = performance_reliability.loc[
    performance_reliability["Variable"] == "MF_UNICMJ_LEFT"
].iloc[0]

assert np.isclose(check_right["ICC_3_1"], 0.87350, atol=1e-4)
assert np.isclose(check_right["CV"], 5.33608, atol=1e-4)
assert np.isclose(check_right["Comparison_p"], 0.22122, atol=1e-4)
assert np.isclose(check_left["ICC_3_1"], 0.91879, atol=1e-4)
assert np.isclose(check_left["CV"], 4.44766, atol=1e-4)

print("Verification checks passed.")

## 11. Interpretation questions

1. Why should ICC and CV be interpreted together rather than separately?
2. Which performance variables met both criteria for acceptable reliability?
3. Did jump height show the same reliability as force, velocity, power, or impulse?
4. Which unilateral asymmetry metric had the highest ICC? Did any exceed .70?
5. Did the favored limb remain consistent between sessions according to kappa?
6. Why is CV misleading for a signed variable with a mean close to zero?
7. What does a non-significant paired comparison tell you? Why does it **not** prove reliability?
8. Compare your results with Table 2 of Pérez-Castilla et al. (2021). How might rounding, normality-test decisions, confidence-interval conventions, and missing preferred-leg metadata explain differences?

## 12. Save the analysis tables

In [ ]:
output_dir = data_dir / "outputs"
output_dir.mkdir(exist_ok=True)

performance_summary.to_csv(output_dir / "performance_summary.csv", index=False)
asymmetry_summary.to_csv(output_dir / "asymmetry_summary.csv", index=False)
performance_reliability.to_csv(output_dir / "performance_reliability.csv", index=False)
asymmetry_reliability.to_csv(output_dir / "unilateral_asymmetry_reliability.csv", index=False)
kappa_results.to_csv(output_dir / "unilateral_asymmetry_kappa.csv", index=False)

print("Saved files:")
for path in sorted(output_dir.glob("*.csv")):
    print(f"- {path}")

## Final note

Reliability is multidimensional. A metric can show a high ICC because players maintain their rank order, yet still have enough within-player noise to produce an unacceptable CV. Conversely, the absence of a significant session difference does not establish reliability. Interpret systematic change, standardized effect size, absolute error, relative reliability, and direction-of-asymmetry agreement together.